Data Preperation

In [ ]:
import json, numpy as np, pandas as pd
from pycocotools import mask as mu

coco = json.load(open("data/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"))

imgs = pd.DataFrame(coco["images"])
anns = pd.DataFrame(coco["annotations"])

imgs["annotator"] = imgs["id"].str.split("-", n=1).str[0]
imgs["obs"]       = imgs["id"].str.split("-", n=1).str[1]          #GROUP KEY
imgs["time"]      = pd.to_datetime(imgs["obs"].str[:14], format="%Y%m%d%H%M%S")
imgs["instrument"] = imgs["obs"].str[14:]

def ann_to_mask(ann, h=2048, w=2048):
    rle = mu.merge(mu.frPyObjects(ann["segmentation"], h, w))
    return mu.decode(rle)                                           #uint8 HxW

def mask_to_counts(binary_mask):                                    #for submission.csv
    rle = mu.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return rle["counts"].decode("ascii")

In [ ]:
#counts images, observations, filaments, annotators and instruments (to tell you how big the dataset is)
print("images (annotation sets):", len(imgs))
print("unique observations:", imgs["obs"].nunique())
print("annotated filaments:", len(anns))
print("annotators:", imgs["annotator"].nunique())
print("\nannotators per observation:")
print(imgs.groupby("obs")["annotator"].nunique().value_counts().sort_index())
print("\ninstruments:")
print(imgs["instrument"].value_counts())
print("\ndate range:", imgs["time"].min(), "->", imgs["time"].max())
print("\nimage sizes:", imgs[["width", "height"]].drop_duplicates().values.tolist())

images (annotation sets): 1154
unique observations: 707
annotated filaments: 8199
annotators: 37

annotators per observation:
annotator
1    411
2    145
3    151
Name: count, dtype: int64

instruments:
instrument
Lh    210
Mh    205
Bh    198
Ch    195
Th    180
Uh    166
Name: count, dtype: int64

date range: 2011-01-09 10:47:34 -> 2022-08-03 07:47:52

image sizes: [[2048, 2048]]


In [ ]:
#how many filaments each image has and how large they are (show the size and imbalance of what must be segmented)
per_img = anns.groupby("image_id").size()
print("filaments per image:\n", per_img.describe())
print("\nimages with zero filaments:", len(imgs) - per_img.shape[0])
print("\nfilament area (pixels):\n", anns["area"].describe())
print("\ncategory counts:\n", anns["category_id"].value_counts())

filaments per image:
 count    1154.000000
mean        7.104853
std         4.375513
min         1.000000
25%         4.000000
50%         7.000000
75%        10.000000
max        26.000000
dtype: float64

images with zero filaments: 0

filament area (pixels):
 count     8199.000000
mean      2119.834980
std       2741.721526
min          9.000000
25%        670.000000
50%       1228.000000
75%       2438.000000
max      37739.000000
Name: area, dtype: float64

category counts:
 category_id
3    3074
2    2590
1    2535
Name: count, dtype: int64


In [ ]:
#divides into 5 folds (by calendar month) - saves that to splits.csv
obs = imgs.drop_duplicates("obs")[["obs", "time", "instrument"]].copy()
obs["block"] = obs["time"].dt.to_period("M").astype(str)

K = 5
rng = np.random.RandomState(42)
sizes = obs["block"].value_counts()
order = sizes.sample(frac=1, random_state=42).sort_values(ascending=False, kind="stable").index  #big blocks first
fold_load = np.zeros(K); block_fold = {}
for b in order:
    f = int(np.argmin(fold_load)); block_fold[b] = f; fold_load[f] += sizes[b]
obs["fold"] = obs["block"].map(block_fold)

def get_split(fold, gap_days=2):
    val = obs[obs.fold == fold]
    tr = obs[obs.fold != fold]
    d = np.abs(tr["time"].values[:, None] - val["time"].values[None, :]).min(axis=1)
    tr = tr[d > np.timedelta64(gap_days, "D")]
    return tr["obs"].tolist(), val["obs"].tolist()

print("observations per fold:\n", obs["fold"].value_counts().sort_index())
print("\ninstrument mix per fold:\n", pd.crosstab(obs["fold"], obs["instrument"]))
for f in range(K):
    tr, va = get_split(f)
    print(f"fold {f}: train obs {len(tr)}, val obs {len(va)}")

out = imgs[["id", "file_name", "obs", "annotator", "time", "instrument"]].merge(obs[["obs", "fold"]], on="obs")
out.to_csv("splits.csv", index=False)

observations per fold:
 fold
0    142
1    142
2    141
3    141
4    141
Name: count, dtype: int64

instrument mix per fold:
 instrument  Bh  Ch  Lh  Mh  Th  Uh
fold                              
0           20  26  20  26  27  23
1           27  29  29  15  28  14
2           20  24  26  33  18  20
3           20  22  27  27  17  28
4           31  24  25  24  20  17
fold 0: train obs 561, val obs 142
fold 1: train obs 564, val obs 142
fold 2: train obs 561, val obs 141
fold 3: train obs 561, val obs 141
fold 4: train obs 562, val obs 141


In [ ]:
#check for data leakage
for f in range(5):
    tr, va = get_split(f)
    assert len(set(tr) & set(va)) == 0, "you leaked it damn"
print("no case appears in both train and val (in any fold)")

no case appears in both train and val (in any fold)


In [13]:
#this is used to speed up training
#instead of fetching labels from the data, its cashed for faster retrieval

import os
os.makedirs("cache/labels", exist_ok=True)

for iid in imgs["id"]:
    lab = np.zeros((2048, 2048), np.uint16)
    for k, r in enumerate(rles_of(iid), 1):
        lab[mu.decode(r) > 0] = k
    np.savez_compressed(f"cache/labels/{iid}.npz", lab=lab)
print("cached", len(os.listdir("cache/labels")), "label maps")

cached 1154 label maps
